# A2J Charter Intervention Accelerator: Mapping Intervenors in SCC Litigation (2011-2025)

**Project Overview:**
This notebook serves as the final code submission for the A2J Charter Intervention Accelerator project. 

## Code 3.1

In [ ]:
!pip install networkx matplotlib pyvis

In [ ]:
import pandas as pd
import networkx as nx
from pyvis.network import Network

print("Building and Rendering the Bipartite Network...")

# 1. Read the edge list generated from the cleaned dataset
df_edges = pd.read_csv("A2J_Network_Edges.csv")
G = nx.Graph()

# 2. Build network nodes based on the edge list
for index, row in df_edges.iterrows():
    org = row['NGO']
    lawyer = row['Counsel']
    weight = row['Weight']
    
    # Initialize nodes and accumulate total appearances
    if not G.has_node(org): G.add_node(org, type='NGO', weight=0)
    if not G.has_node(lawyer): G.add_node(lawyer, type='Counsel', weight=0)
    
    G.nodes[org]['weight'] += weight
    G.nodes[lawyer]['weight'] += weight
    
    # Build edges
    G.add_edge(org, lawyer, weight=weight)

# 3. Filter core players (NGO >= 3 times, Lawyer >= 2 times)
valid_orgs = [n for n, attr in G.nodes(data=True) if attr['type'] == 'NGO' and attr['weight'] >= 3]
valid_lawyers = set()
for org in valid_orgs:
    for neighbor in G.neighbors(org):
        if G.nodes[neighbor]['weight'] >= 2:
            valid_lawyers.add(neighbor)

valid_nodes = set(valid_orgs).union(valid_lawyers)
G_core = G.subgraph(valid_nodes).copy()

print(f"Network built! Core NGOs: {len(valid_orgs)}, Core Counsel: {len(valid_lawyers)}")

# 4. PyVis Visualization
net = Network(height="850px", width="100%", bgcolor="#fdfdfd", font_color="#333333", cdn_resources='remote', select_menu=True)
sorted_nodes = sorted(G_core.nodes(data=True), key=lambda x: str(x[0]))

for node, attr in sorted_nodes:
    node_weight = attr['weight']
    safe_label = str(node).replace("<", "").replace(">", "").replace("&", "and")
    
    if attr['type'] == 'NGO':
        hover = f"{safe_label}\nType: NGO\nTotal Appearances: {node_weight}"
        net.add_node(node, label=safe_label, size=node_weight * 2 + 15, title=hover, color="#1B4D3E", shape="dot")
    else:
        hover = f"{safe_label}\nType: Counsel\nTotal Appearances: {node_weight}"
        net.add_node(node, label=safe_label, size=node_weight * 3 + 10, title=hover, color="#8B1A1A", shape="triangle")

for u, v, attr in G_core.edges(data=True):
    edge_weight = attr['weight']
    hover = f"Counsel-Client Relationship\nRepresented {edge_weight} times"
    net.add_edge(u, v, value=edge_weight, title=hover, color="#999999")

net.force_atlas_2based(gravity=-60, central_gravity=0.01, spring_length=150, spring_strength=0.03, damping=0.9, overlap=0.5)
net.write_html("A2J_Counsel_NGO_Map.html")

print("✅ Complete: Interactive Web Map Generated (A2J_Counsel_NGO_Map.html)")

## Code 3.2

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# 1. Use Seaborn and Matplotlib to create a heatmap visualization of the full Bipartite Matrix of NGOs vs. Charter sections, which will allow us to visually analyze the relationships between specific intervenor organizations and the Charter sections they are associated with in SCC litigation. We will customize the appearance of the heatmap to ensure that it is clear, informative, and visually appealing, making it easier for viewers to identify patterns and key players in the network of NGOs and Charter sections.
# ==========================================
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial'] 
plt.rcParams['axes.unicode_minus'] = False 

# ==========================================
# 2. Load the full Bipartite Matrix CSV that we generated in the previous step, which contains the frequency of connections between each NGO and each Charter section. This matrix will be used as the data source for our heatmap visualization, allowing us to analyze the relationships between NGOs and Charter sections in a visual format that highlights patterns and key players in SCC litigation.
# ==========================================
print("Loading the full Bipartite Matrix CSV...")
try:
    file_path = "A2J_NGO_vs_All_Sections_Matrix.csv"
    # Load the full matrix CSV that we generated in the previous step, which contains the frequency of connections between each NGO and each Charter section. This matrix will be used as the data source for our heatmap visualization, allowing us to analyze the relationships between NGOs and Charter sections in a visual format that highlights patterns and key players in SCC litigation.
    df_matrix = pd.read_csv(file_path, index_col='NGO')
    print("-> Full matrix loaded successfully.")
except FileNotFoundError:
    print(f"❌ error '{file_path}' not found. Please ensure that the file exists and the path is correct.")
    # Exit the script with a non-zero status code to indicate that an error occurred, which can be useful for debugging and for automated workflows that check for successful execution. This will prevent the rest of the code from running if the necessary data file is not available, avoiding further errors and providing a clear indication of what went wrong.
    import sys; sys.exit(1) 

# ==========================================
# 3. Data processing for the heatmap: We will perform several steps to prepare the data for visualization, including replacing zero values with NaN for better aesthetics, removing columns that could skew the color scale, ordering the columns by frequency, and limiting the number of NGOs displayed to focus on the most active ones. These steps will help to create a clearer and more informative heatmap that highlights the key relationships between NGOs and Charter sections in SCC litigation.
# ==========================================
print("Processing data for ALL sections...")

# a) Replace zero values with NaN to improve the aesthetics of the heatmap. This will allow us to use a color scheme that emphasizes the presence of connections between NGOs and Charter sections, while leaving the absence of connections (zero values) as blank spaces in the heatmap, making it easier to visually identify patterns and clusters of activity.
df_plot = df_matrix.replace(0, np.nan)

# b) Remove the 'Total_Appearances' column from the DataFrame before plotting, as it is not a Charter section and could skew the color scale of the heatmap. This column is useful for sorting and analysis, but it should not be included in the heatmap itself, as it does not represent a specific Charter section and could distort the visual representation of the data.
if 'Total_Appearances' in df_plot.columns:
    df_plot = df_plot.drop(columns=['Total_Appearances'])
if 's.Unspecified' in df_plot.columns: 
    df_plot = df_plot.drop(columns=['s.Unspecified'])
    
# c) Order the columns (Charter sections) by their total frequency across all NGOs, so that the most frequently mentioned sections appear on the left side of the heatmap. This will help to visually emphasize the most relevant Charter sections in SCC litigation and make it easier for viewers to identify which sections are most commonly associated with NGO interventions.
section_counts = df_plot.sum(axis=0) 
sorted_sections = section_counts.sort_values(ascending=False).index
df_plot = df_plot[sorted_sections]

# d) To prevent the heatmap from becoming too cluttered and unreadable due to the large number of NGOs, we will limit the number of NGOs displayed to the top 50 based on their total appearances across all Charter sections. This will allow us to focus on the most active and influential NGOs in the dataset, while still providing a comprehensive view of their involvement with various Charter sections in SCC litigation. By showing only the top 50 NGOs, we can ensure that the heatmap remains visually clear and interpretable, while still highlighting key players and their associated Charter sections.
df_plot = df_plot.head(50) 

print(f"-> Drawing Heatmap for {len(df_plot)} NGOs vs. {len(df_plot.columns)} Charter Sections.")

# ==========================================
# 4. Creating the heatmap with Seaborn, using a visually appealing color scheme and adjusting the figure size dynamically based on the number of NGOs being plotted. This will ensure that the heatmap is large enough to be readable without being excessively large, and that the most active NGOs and their associated Charter sections are clearly visible and easy to interpret.
# ==========================================
# Adjust the figure size dynamically based on the number of NGOs being plotted, to ensure that the heatmap is large enough to be readable without being excessively large. The width is set to 50 inches to accommodate a large number of columns (Charter sections), while the height is calculated based on the number of NGOs, with a base height plus an additional amount for each NGO to ensure that the rows are not too cramped.
fig_height = len(df_plot) * 0.35 + 5 
plt.figure(figsize=(50, fig_height)) 

# Use the 'rocket_r' colormap from Seaborn, which is a reversed version of the 'rocket' colormap. This color scheme will provide a visually appealing gradient that emphasizes higher frequencies with warmer colors, while lower frequencies will be represented with cooler colors. The reversed version ensures that the most active NGOs (with the highest counts) will stand out more prominently in the heatmap, making it easier to identify key players and their associated Charter sections in SCC litigation.
cmap = sns.cm.rocket_r 

# Create the heatmap using Seaborn's heatmap function, with the processed DataFrame 'df_plot' as the data source. The heatmap will use the specified colormap, and we will disable annotations to keep the visual clean. The linewidths parameter will add lines between the cells for better separation, and the cbar_kws parameter will add a label to the color bar indicating that it represents the number of interventions. The square parameter is set to False to allow for rectangular cells, which can help accommodate a large number of columns without making the cells too small.
sns.heatmap(
    df_plot,
    cmap=cmap, 
    annot=False, 
    fmt=".0f", 
    linewidths=.8, 
    cbar_kws={'label': 'Number of Interventions'}, 
    square=False 
)

# 
plt.title(
    "A2J Charter Accelerator: Full Bipartite Strategic Matrix\nNGOs vs. ALL Charter Sections (2011-2025)",
    fontsize=24,
    fontweight='bold',
    pad=30
)
plt.ylabel("NGO (Ordered by Total Activity)", fontsize=18)
plt.xlabel("Charter Section (Ordered by Frequency)", fontsize=18)
plt.xticks(rotation=45, ha='right', fontsize=12) 
plt.yticks(fontsize=10)

plt.tight_layout()

# Save the heatmap as a high-resolution PNG file with a descriptive name, so that it can be easily shared and referenced in reports or presentations about the involvement of NGOs in SCC Charter litigation across all sections. The file will be saved with a DPI of 300 to ensure that it looks crisp and professional when printed or displayed on high-resolution screens.
heatmap_file = "A2J_NGO_All_Sections_Heatmap.png"
plt.savefig(heatmap_file, dpi=300, bbox_inches='tight') 

plt.show()

print(f"\n✅ FULL HEATMAP SUCCESSFULLY GENERATED!")
print(f"   -> An extremely detailed strategic map has been saved as: {heatmap_file}")

## Code 3.3

### 3.3.1 - for NGO:

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Set up the visual style and font for the heatmap to ensure that it is clear, professional, and visually appealing. We will use Seaborn's whitegrid style for a clean background, and we will set the font to Arial for better readability. This will help to create a heatmap that is easy to read and interpret, while also maintaining a polished and professional appearance for presentations or reports on the involvement of NGOs in SCC Charter litigation.
plt.rcParams['font.sans-serif'] = ['Arial']
sns.set_theme(style="whitegrid")

# 2. Load the full Bipartite Matrix CSV that we generated in the previous step, which contains the frequency of connections between each NGO and each Charter section. This matrix will be used as the data source for our heatmap visualization, allowing us to analyze the relationships between NGOs and Charter sections in a visual format that highlights patterns and key players in SCC litigation.
print("Loading NGO Matrix CSV...")
try:
    df_matrix = pd.read_csv("A2J_NGO_vs_All_Sections_Matrix.csv", index_col='NGO')
except FileNotFoundError:
    print("❌ cannot find 'A2J_NGO_vs_All_Sections_Matrix.csv'")
    import sys; sys.exit(1)

# Cleaning up the DataFrame by removing columns that are not relevant for the heatmap visualization, such as 'Total_Appearances' and any 'Unspecified' columns. These columns could skew the color scale of the heatmap and do not represent specific Charter sections, so we will drop them to ensure that the heatmap accurately reflects the relationships between NGOs and the specific Charter sections they are associated with in SCC litigation.
cols_to_drop = ['Total_Appearances', 's.Unspecified', 'Unspecified']
for c in cols_to_drop:
    if c in df_matrix.columns:
        df_matrix = df_matrix.drop(columns=[c])

# 3. Find the Top 10 Charter Sections
top_10_sections = df_matrix.sum().sort_values(ascending=False).head(10).index.tolist()

# 4. Marginal bar charts for the Top 10 Charter Sections, showing the top NGOs associated with each section. We will create a grid of bar charts, with one chart for each of the top 10 Charter sections, and we will customize the appearance of the charts to make them visually appealing and easy to interpret. This will allow us to highlight the most active NGOs for each of the most frequently mentioned Charter sections in SCC litigation.
fig, axes = plt.subplots(2, 5, figsize=(32, 14))
axes = axes.flatten()
fig.subplots_adjust(wspace=0.8, hspace=0.3)

for i, section in enumerate(top_10_sections):
    ax = axes[i]
    sec_data = df_matrix[section].sort_values(ascending=False).head(10).reset_index()
    sec_data.columns = ['NGO', 'Interventions']
    sec_data = sec_data[sec_data['Interventions'] > 0]
    
    # Color setting
    sns.barplot(
        x='Interventions', y='NGO', data=sec_data, 
        palette="Blues_r", ax=ax
    )
    
    # Color setting
    ax.set_title(f"{section}", fontsize=18, fontweight='bold', color='#8B1A1A', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(axis='y', labelsize=12) 
    ax.tick_params(axis='x', labelsize=10)
    
    # Add value labels to the bars
    for p in ax.patches:
        width = p.get_width()
        if width > 0:
            ax.annotate(f'{int(width)}', 
                        (width, p.get_y() + p.get_height() / 2.), 
                        ha='left', va='center', xytext=(4, 0), textcoords='offset points',
                        fontsize=12, fontweight='bold', color='#333333')

# Color setting
plt.suptitle(
    "A2J Strategic Radar: Top 10 NGOs across the Top 10 Litigated Charter Sections (2011-2025)", 
    fontsize=28, fontweight='bold', color='#003366', y=1.02
)

# Marginal note about the data source and methodology
plt.tight_layout(rect=[0, 0, 1, 0.98])

# Save the chart as a high-resolution PNG file with a descriptive name, so that it can be easily shared and referenced in reports or presentations about the involvement of NGOs in SCC Charter litigation across the most frequently mentioned sections. The file will be saved with a DPI of 300 to ensure that it looks crisp and professional when printed or displayed on high-resolution screens.
chart_filename = "A2J_Dashboard_NGOs.png"
plt.savefig(chart_filename, dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ NGO Dashboard saved successfully as: {chart_filename}")

### 3.3.2 - for Counsels

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import ast
import re

# ==========================================
# 1. Load the case data with charter sections and build a new matrix that focuses on the relationships between individual counsel and Charter sections. We will read the dataset, extract the relevant information about counsel and Charter sections, and create a new DataFrame that captures these relationships. We will then generate a crosstab matrix from this DataFrame, which will show the frequency of connections between each counsel and each Charter section across all cases in the dataset. Finally, we will save this matrix as a CSV file for further analysis and reporting on the involvement of individual lawyers in SCC Charter litigation.
# ==========================================
print("Building the pristine Counsel Matrix from raw case data...")
try:
    df_cases = pd.read_csv("A2J_Cases_with_Charter_Sections.csv")
except FileNotFoundError:
    print("❌ Cannot find 'A2J_Cases_with_Charter_Sections.csv'")
    import sys; sys.exit(1)

records = []
for index, row in df_cases.iterrows():
    text = row['Intervenors_and_Counsel']
    sections_str = row['All_Sections']
    
    if pd.isna(text) or "No Intervenors" in text or pd.isna(sections_str): continue
        
    try:
        sections = ast.literal_eval(sections_str)
    except:
        sections = []
        
    lines = str(text).split('\n')
    for line in lines:
        if '->' in line:
            parts = line.split('->')
            raw_org = parts[0].replace('[Intervener]', '').strip()
            raw_counsel = parts[1].strip()
            
            # Filter out any organizations that contain keywords associated with government institutions, as we want to focus on individual lawyers and not official government representatives. This will help us to create a more accurate and meaningful analysis of counsel involvement in SCC Charter litigation, by excluding any entries that are likely to represent institutional actors rather than individual lawyers.
            if any(kw in raw_org.lower() for kw in state_keywords):
                continue
            
            if raw_counsel and raw_counsel != "None listed":
                cleaned_c_str = clean_counsel_string(raw_counsel)
                lawyers = [clean_name(l) for l in cleaned_c_str.split(',')]
                
                for lawyer in lawyers:
                    if not lawyer: continue
                    for sec in sections:
                        if sec != 's.Unspecified' and sec != 'Unspecified':
                            records.append({'Counsel': lawyer, 'Charter_Section': sec})

df_counsel = pd.DataFrame(records)
df_matrix_counsel = pd.crosstab(df_counsel['Counsel'], df_counsel['Charter_Section'])
df_matrix_counsel.to_csv("A2J_Counsel_vs_All_Sections_Matrix.csv")
print("✅ Pristine Counsel Matrix generated and saved!")

# ==========================================
# 2. Load the Counsel vs. Charter Sections matrix that we generated in the previous step, which contains the frequency of connections between each counsel and each Charter section. This matrix will be used as the data source for our heatmap visualization, allowing us to analyze the relationships between individual lawyers and the specific Charter sections they are associated with in SCC litigation.
# ==========================================
print("Drawing Counsel Dashboard...")

plt.rcParams['font.sans-serif'] = ['Arial']
sns.set_theme(style="whitegrid")

# Find the Top 10 Charter Sections based on counsel involvement
top_10_sections_c = df_matrix_counsel.sum().sort_values(ascending=False).head(10).index.tolist()

# Margin Setting
fig, axes = plt.subplots(2, 5, figsize=(32, 14))
axes = axes.flatten()
fig.subplots_adjust(wspace=0.8, hspace=0.3) 

for i, section in enumerate(top_10_sections_c):
    ax = axes[i]
    sec_data = df_matrix_counsel[section].sort_values(ascending=False).head(10).reset_index()
    sec_data.columns = ['Counsel', 'Interventions']
    sec_data = sec_data[sec_data['Interventions'] > 0]
    
    sns.barplot(
        x='Interventions', y='Counsel', data=sec_data, 
        palette="Blues_r", ax=ax
    )
    
    ax.set_title(f"{section}", fontsize=18, fontweight='bold', color='#8B1A1A', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(axis='y', labelsize=12)
    ax.tick_params(axis='x', labelsize=10)
    
    for p in ax.patches:
        width = p.get_width()
        if width > 0:
            ax.annotate(f'{int(width)}', 
                        (width, p.get_y() + p.get_height() / 2.), 
                        ha='left', va='center', xytext=(4, 0), textcoords='offset points',
                        fontsize=12, fontweight='bold', color='#333333')

plt.suptitle(
    "A2J Strategic Radar: Top 10 Individual Counsel across the Top 10 Litigated Charter Sections (2011-2025)", 
    fontsize=28, fontweight='bold', color='#003366', y=1.02
)

plt.tight_layout(rect=[0, 0, 1, 0.98])

chart_filename_counsel = "A2J_Dashboard_Counsel.png"
plt.savefig(chart_filename_counsel, dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Counsel Dashboard saved successfully as: {chart_filename_counsel}")

# Appendix A: Data Source Selection and Web-Scraping Strategy

### A.1 Attempting of Using the A2AJ Database to Extract Intervenors

In [ ]:
!pip install datasets pandas

In [ ]:
import pandas as pd
import re
import os
from datasets import load_dataset
from dotenv import load_dotenv
from openai import OpenAI

# load environment variables from .env file (especially OPENAI_API_KEY)
load_dotenv()

# initialize OpenAI client
client = OpenAI()

# ==========================================
# 1. Data Loading 
# ==========================================
def load_scc_data(dataset_name="a2aj/canadian-case-law", folder="SCC"):
    print("Loading specifically the SCC subset...")
    try:
        dataset = load_dataset(dataset_name, data_dir=folder, split="train")
        df = dataset.to_pandas()
        print(f"Successfully loaded {len(df)} SCC cases.")
        return df
    except Exception as e:
        print(f"Error loading data: {e}")
        return None

# ==========================================
# 2. Filtering for Charter Cases
# ==========================================
def filter_charter_cases(df):
    print("Filtering for Charter litigation cases (2010-2024)...")
    df['year'] = pd.to_datetime(df['document_date_en'], errors='coerce', utc=True).dt.year
    timeframe_df = df[(df['year'] >= 2010) & (df['year'] <= 2024)].copy()

    charter_pattern = re.compile(r'\b(?:Charter of Rights and Freedoms|s\.?\s*15|s\.?\s*7)\b', re.IGNORECASE)
    
    charter_df = timeframe_df[
        timeframe_df['unofficial_text_en'].astype(str).str.contains(charter_pattern, regex=True, na=False)
    ].copy()

    print(f"Found {len(charter_df)} Charter-related cases within the timeframe.")
    return charter_df

# ==========================================
# 3. Intervenor Extraction (GenAI Model)
# ==========================================
import json 

def extract_metadata_genai(text):
    """
    This function takes the headnote text of an SCC case and uses a GenAI model to extract intervenor and counsel information.
    """
    # To prevent token overflow, we will only send the first 40,000 characters of the headnote text to the model.
    header_text = str(text)[:40000] 
    
    try:
        response = client.chat.completions.create(
            model="gpt-5-nano",
            response_format={ "type": "json_object" }, 
            messages=[
                {
                    "role": "system", 
                    "content": (
                        "You are a highly precise legal data extraction API analyzing SCC judgment headers. "
                        "Your ONLY job is to output a strictly formatted JSON object. "
                        "\n\nCRITICAL RULES:"
                        "\n1. BE EXHAUSTIVE: You MUST extract EVERY single intervenor and EVERY single counsel. Do not truncate. If there are 10 counsel for one org, list all 10."
                        "\n2. EXCLUDE AGENTS: Do NOT extract 'Ottawa Agents', 'Agents', or 'Correspondents'. ONLY extract 'Counsel', 'Solicitors', or 'Avocats'."
                        "\n3. MAPPING: Accurately link counsel to their specific organization."
                        "\n\nYou MUST return a JSON object with a single key 'intervenors', which contains a list of objects. "
                        "Example JSON structure: "
                        "{\"intervenors\": [{\"organization\": \"Org Name\", \"counsel\": [\"Counsel A\", \"Counsel B\"]}, {\"organization\": \"Org 2\", \"counsel\": []}]}"
                    )
                },
                {
                    "role": "user", 
                    "content": f"Extract the data from this text into the required JSON format:\n\n{header_text}"
                }
            ]
        )
        
        # extrace the JSON string from the model's response
        json_str = response.choices[0].message.content.strip()
        
        # parse the JSON string into a Python dictionary
        data = json.loads(json_str)
        result_lines = []
        
        for item in data.get('intervenors', []):
            org = item.get('organization', 'Unknown Org')
            counsel_list = item.get('counsel', [])
            
            if not counsel_list:
                counsel_str = "None/Data Missing"
            else:
                counsel_str = ", ".join(counsel_list)
                
            result_lines.append(f"{org}: {counsel_str}")
            
        return " | ".join(result_lines) if result_lines else "None Found"

    except json.JSONDecodeError:
        return "API Error: Failed to parse JSON"
    except Exception as e:
        return f"API Error: {e}"

# ==========================================
# 4. Main Execution Pipeline
# ==========================================
def main():
    df = load_scc_data()
    
    if df is not None:
        charter_cases_df = filter_charter_cases(df)
        
        print("\n--- Testing GenAI Extraction (Sample of 3 cases) ---")
        # test the GenAI extraction on a small sample of cases to verify output format and accuracy
        sample_df = charter_cases_df.head(3).copy() 
        
        # script to apply the GenAI extraction function to the sample cases and store results in a new column
        sample_df['GenAI_Extracted_Data'] = sample_df['unofficial_text_en'].apply(extract_metadata_genai)
        
        # print the results for the sample cases to verify correctness and formatting
        for index, row in sample_df.iterrows():
            print(f"\nCase Name: {row['name_en']}")
            print(f"Extraction Result:\n{row['GenAI_Extracted_Data']}")
            print("-" * 40)
        
        print("\nInitial pipeline execution complete. Ready for Tuesday submission!")

if __name__ == "__main__":
    main()

### A.2: The Pivot to SCC Official Website

##### Test of first 3 cases:

In [ ]:
import pandas as pd
import re
import time
import requests
import json
from bs4 import BeautifulSoup
from datasets import load_dataset
from dotenv import load_dotenv
from openai import OpenAI

# ==========================================
# 0. Setup & Authentication
# ==========================================
load_dotenv()
client = OpenAI()

# ==========================================
# 1. Data Indexing (A2AJ Dataset -> 2011-2025 Charter Cases)
# ==========================================
def load_and_filter_index():
    print("Loading A2AJ dataset as master index...")
    try:
        dataset = load_dataset("a2aj/canadian-case-law", data_dir="SCC", split="train")
        df = dataset.to_pandas()
        
        df['year'] = pd.to_datetime(df['document_date_en'], errors='coerce', utc=True).dt.year
        timeframe_df = df[(df['year'] >= 2011) & (df['year'] <= 2025)].copy()

        charter_pattern = re.compile(r'\b(?:Charter of Rights and Freedoms|s\.?\s*15|s\.?\s*7)\b', re.IGNORECASE)
        charter_df = timeframe_df[
            timeframe_df['unofficial_text_en'].astype(str).str.contains(charter_pattern, regex=True, na=False)
        ].copy()
        
        print(f"Index built: Found {len(charter_df)} Charter cases (2011-2025).")
        return charter_df
    except Exception as e:
        print(f"Error loading index: {e}")
        return None

# ==========================================
# 2. Extract 5-Digit Docket Number
# ==========================================
def extract_docket_number(text):
    """extract the 5-digit docket number from the headnote text using regex."""
    match = re.search(r'(?:File No\.|No du greffe|dossier)\s*[:\.]?\s*(\d{5})', str(text), re.IGNORECASE)
    if match: return match.group(1)
    fallback = re.search(r'\b(\d{5})\b', str(text)[:1000])
    return fallback.group(1) if fallback else None

# ==========================================
# 3. Scrape Clean Counsel Text from SCC Registry
# ==========================================
def scrape_clean_counsel_text(docket_number):
    """visit the SCC registry page for the given docket number and extract the clean text containing counsel information."""
    if not docket_number: return ""
    
    url = f"https://www.scc-csc.ca/cases-dossiers/search-recherche/{docket_number}/"
    time.sleep(1.2) # polite scraping: 1.2 second delay between requests to avoid hitting rate limits or overwhelming the server
    
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'html.parser')
        page_text = soup.get_text(separator='\n', strip=True)
        
        # scrape from "Main Parties"
        start_idx = page_text.find("Main parties")
        
        # if "Main parties" is not found, try alternative keywords to ensure we capture the counsel information
        if start_idx == -1: 
            start_idx = page_text.find("Party information")
        if start_idx == -1:
            start_idx = page_text.find("Counsel\nParty:")
            
        if start_idx != -1:
            # scrape a large chunk of text starting from the identified index to ensure we capture all counsel information, but limit to 15,000 characters to avoid token overflow
            return page_text[start_idx : start_idx + 15000] 
        return ""
    except Exception as e:
        print(f"Scraping error on {docket_number}: {e}")
        return ""

# ==========================================
# 4. GenAI JSON Extraction (Cross-referencing Roles)
# ==========================================
def parse_with_genai(clean_text):
    """complex GenAI prompt to extract intervenor and counsel information, ensuring we only capture intervenors and their counsel, while excluding appellants/respondents and agents."""
    if not clean_text: return "No Data Scraped"
    
    try:
        response = client.chat.completions.create(
            model="gpt-5-nano",
            response_format={ "type": "json_object" },
            messages=[
                {
                    "role": "system", 
                    "content": (
                        "You are a strict legal data parser analyzing SCC registry text. "
                        "The text usually begins with a 'Main parties' section defining Appellants and Respondents, "
                        "followed later by a Counsel section detailing the lawyers for each party. "
                        "\n\nCRITICAL RULES:"
                        "\n1. TARGET INTERVENERS ONLY: Cross-reference the names. Do NOT extract any party listed as an 'Appellant' or 'Respondent' in the Main parties section. ONLY extract 'Interveners' (third-party organizations, NGOs, clinics, or governments)."
                        "\n2. IGNORE AGENTS: Only extract names explicitly listed under 'Counsel', 'Solicitors', or 'Names' (if in the counsel block). Do NOT extract 'Agent'."
                        "\n3. BE EXHAUSTIVE: List all counsel for each intervener."
                        "\n4. Return a JSON object with a single key 'intervenors'. "
                        "Format: {\"intervenors\": [{\"party_name\": \"Name\", \"counsel\": [\"Counsel A\", \"Counsel B\"]}]}"
                    )
                },
                {
                    "role": "user", 
                    "content": f"Text to parse:\n{clean_text}"
                }
            ]
        )
        
        data = json.loads(response.choices[0].message.content.strip())
        
        # extract and format the results into a readable string for display
        results = []
        for item in data.get('intervenors', []):
            party = item.get('party_name', 'Unknown')
            counsel = ", ".join(item.get('counsel', [])) if item.get('counsel') else "None listed"
            results.append(f"[Intervener] {party} -> {counsel}")
            
        return "\n".join(results) if results else "No Intervenors found in this case."
        
    except Exception as e:
        return f"API Error: {e}"

# ==========================================
# 5. Main Pipeline Execution & Data Export
# ==========================================
def main():
    df = load_and_filter_index()
    
    if df is not None:
        print("\n--- Running Final Extraction Pipeline (Sample Mode) ---")
        
        # use a small sample of cases for the PoC to ensure we can run end-to-end without hitting API limits or taking too long during grading
        test_cases = ["Loyola High School", "Sanghera", "Quebec.*Canada"]
        mask = df['name_en'].str.contains('|'.join(test_cases), na=False, regex=True)
        sample_df = df[mask].head(3).copy()
        
        # create a list to hold the final structured data for all cases
        final_dataset = []
        
        for index, row in sample_df.iterrows():
            case_name = row['name_en']
            print(f"\nProcessing Case: {case_name}")
            
            docket = extract_docket_number(row['unofficial_text_en'])
            print(f"Docket: {docket}")
            
            # 1. scrape the clean counsel text from the SCC registry page for this case using the extracted docket number
            clean_text = scrape_clean_counsel_text(docket)
            
            # 2. use the GenAI model to parse the clean text and extract intervenor and counsel information
            extracted_data = parse_with_genai(clean_text)
            
            print(f"[ Extracted ]\n{extracted_data}\n" + "-"*40)
            
            # 3. package the extracted data into a structured format for this case and append to the final dataset list
            final_dataset.append({
                'Docket_Number': docket,
                'Case_Name': case_name,
                'Intervenors_and_Counsel': extracted_data,
                'SCC_Registry_URL': row['url_en']
            })
            
        # ==========================================
        # 6. Proof of Concept: Data Delivery
        # ==========================================
        print("\n--- Creating Structured Dataset ---")
        # transfer results to Pandas DataFrame
        results_df = pd.DataFrame(final_dataset)
        
        # print the results in a clean table format for grading review
        display(results_df) 
        
        # output the results to a CSV file as the tangible deliverable for the PoC
        output_filename = "A2J_Intervenors_Sample.csv"
        results_df.to_csv(output_filename, index=False)
        print(f"\n✅ SUCCESS! Extracted data successfully saved to: {output_filename}")
        print("Initial code execution complete. Ready for grading!")

if __name__ == "__main__":
    main()

#### Running the full 402 cases between 2011-2025

In [ ]:
import os
import pandas as pd
import re
import time
import requests
import json
from bs4 import BeautifulSoup
from datasets import load_dataset
from dotenv import load_dotenv
from openai import OpenAI

# ==========================================
# 0. Setup & Authentication
# ==========================================
load_dotenv()
client = OpenAI()

# ==========================================
# 1. Data Indexing (A2AJ Dataset -> 2011-2025 Charter Cases)
# ==========================================
def load_and_filter_index():
    print("Loading A2AJ dataset as master index...")
    try:
        dataset = load_dataset("a2aj/canadian-case-law", data_dir="SCC", split="train")
        df = dataset.to_pandas()
        
        df['year'] = pd.to_datetime(df['document_date_en'], errors='coerce', utc=True).dt.year
        timeframe_df = df[(df['year'] >= 2011) & (df['year'] <= 2025)].copy()

        charter_pattern = re.compile(r'\b(?:Charter of Rights and Freedoms|s\.?\s*15|s\.?\s*7)\b', re.IGNORECASE)
        charter_df = timeframe_df[
            timeframe_df['unofficial_text_en'].astype(str).str.contains(charter_pattern, regex=True, na=False)
        ].copy()
        
        print(f"Index built: Found {len(charter_df)} Charter cases (2011-2025).")
        return charter_df
    except Exception as e:
        print(f"Error loading index: {e}")
        return None

# ==========================================
# 2. Extract 5-Digit Docket Number
# ==========================================
def extract_docket_number(text):
    """extract the 5-digit docket number from the text"""
    match = re.search(r'(?:File No\.|No du greffe|dossier)\s*[:\.]?\s*(\d{5})', str(text), re.IGNORECASE)
    if match: return match.group(1)
    fallback = re.search(r'\b(\d{5})\b', str(text)[:1000])
    return fallback.group(1) if fallback else None

# ==========================================
# 3. Scrape Clean Counsel Text from SCC Registry
# ==========================================
def scrape_clean_counsel_text(docket_number):
    """access the SCC case page and extract the clean text starting from Main parties, which includes both party definitions and counsel listings"""
    if not docket_number: return ""
    
    url = f"https://www.scc-csc.ca/cases-dossiers/search-recherche/{docket_number}/"
    time.sleep(1.2) # polite scraping
    
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'html.parser')
        page_text = soup.get_text(separator='\n', strip=True)
        
        # 🎯 Move the knife up: start slicing from "Main parties" so the AI can see the official role classifications
        start_idx = page_text.find("Main parties")
        
        # Defensive programming: if Main parties is not found, try alternative keywords
        if start_idx == -1: 
            start_idx = page_text.find("Party information")
        if start_idx == -1:
            start_idx = page_text.find("Counsel\nParty:")
            
        if start_idx != -1:
            # Slice 15000 characters to ensure we capture the long list of parties and the subsequent counsel listings, while being extremely token-efficient
            return page_text[start_idx : start_idx + 15000] 
        return ""
    except Exception as e:
        print(f"Scraping error on {docket_number}: {e}")
        return ""

# ==========================================
# 4. GenAI JSON Extraction (Cross-referencing Roles)
# ==========================================
def parse_with_genai(clean_text):
    """Use gpt-5-nano to force JSON output, eliminating hallucinations and omissions. Cross-reference the Main parties list to exclude Appellants and Respondents, extracting only Intervenors and their counsel."""
    if not clean_text: return "No Data Scraped"
    
    try:
        response = client.chat.completions.create(
            model="gpt-5-nano",
            response_format={ "type": "json_object" },
            messages=[
                {
                    "role": "system", 
                    "content": (
                        "You are a strict legal data parser analyzing SCC registry text. "
                        "The text usually begins with a 'Main parties' section defining Appellants and Respondents, "
                        "followed later by a Counsel section detailing the lawyers for each party. "
                        "\n\nCRITICAL RULES:"
                        "\n1. TARGET INTERVENERS ONLY: Cross-reference the names. Do NOT extract any party listed as an 'Appellant' or 'Respondent' in the Main parties section. ONLY extract 'Interveners' (third-party organizations, NGOs, clinics, or governments)."
                        "\n2. IGNORE AGENTS: Only extract names explicitly listed under 'Counsel', 'Solicitors', or 'Names' (if in the counsel block). Do NOT extract 'Agent'."
                        "\n3. BE EXHAUSTIVE: List all counsel for each intervener."
                        "\n4. Return a JSON object with a single key 'intervenors'. "
                        "Format: {\"intervenors\": [{\"party_name\": \"Name\", \"counsel\": [\"Counsel A\", \"Counsel B\"]}]}"
                    )
                },
                {
                    "role": "user", 
                    "content": f"Text to parse:\n{clean_text}"
                }
            ]
        )
        
        data = json.loads(response.choices[0].message.content.strip())
        
        # Extract results and beautify the output format
        results = []
        for item in data.get('intervenors', []):
            party = item.get('party_name', 'Unknown')
            counsel = ", ".join(item.get('counsel', [])) if item.get('counsel') else "None listed"
            results.append(f"[Intervener] {party} -> {counsel}")
            
        return "\n".join(results) if results else "No Intervenors found in this case."
        
    except Exception as e:
        return f"API Error: {e}"

# ==========================================
# 5. Main Pipeline Execution & Data Export (FULL RUN)
# ==========================================
def main():
    df = load_and_filter_index()
    
    if df is not None:
        # use the full dataset for the final run, but keep the sample run above for demonstration and grading purposes
        target_df = df.copy() 
        total_cases = len(target_df)
        
        print(f"\n--- Running FULL Extraction Pipeline on {total_cases} cases ---")
        
        output_filename = "A2J_Intervenors_Full_Dataset.csv"
        
        # if the output file already exists from a previous run, delete it to avoid appending to old data. This ensures each run starts fresh and the results are clean.
        if os.path.exists(output_filename):
            os.remove(output_filename)
            print(f"🧹 Deleted old {output_filename} to start a fresh run.")
        
        # add a progress counter to keep track of how many cases have been processed in real-time
        for i, (index, row) in enumerate(target_df.iterrows(), 1):
            case_name = row['name_en']
            print(f"\n[{i}/{total_cases}] Processing: {case_name}")
            
            docket = extract_docket_number(row['unofficial_text_en'])
            
            if not docket:
                print("⚠️ Skipping: No Docket Number found in text.")
                continue
                
            # 1. Scrape clean text from SCC registry
            clean_text = scrape_clean_counsel_text(docket)
            
            # 2. Parse with GenAI to extract Intervenors and their counsel, while cross-referencing roles to exclude Appellants and Respondents
            extracted_data = parse_with_genai(clean_text)
            
            # add a quick count of how many interveners were extracted for this case, to provide more insight into the results as we go
            intervener_count = str(extracted_data).count("[Intervener]")
            print(f"-> Extracted {intervener_count} intervener(s).")
            
            # 3. Pack the results into a structured dictionary for this case, which will be appended to the final dataset. This structure allows for easy conversion to DataFrame and CSV later on.
            case_data = {
                'Docket_Number': docket,
                'Case_Name': case_name,
                'Intervenors_and_Counsel': extracted_data,
                'SCC_Registry_URL': row['url_en']
            }
            
            # 4. append the case data to the CSV file immediately after processing each case, to ensure that we have a persistent record of results even if the program crashes midway. This way, we won't lose all progress if something goes wrong, and we can always resume from the last saved case.
            temp_df = pd.DataFrame([case_data])
            temp_df.to_csv(output_filename, mode='a', header=not os.path.exists(output_filename), index=False)
            
        print(f"\n✅ FULL RUN SUCCESS! All {total_cases} cases processed.")
        print(f"Data safely saved to: {output_filename}")

if __name__ == "__main__":
    main()

# Appendix B: Data Cleaning

In [ ]:
!pip install datasets

In [ ]:
import pandas as pd
import re
import ast
from datasets import load_dataset

print("🚀 Starting the Unified A2J Data ETL Pipeline...")

# ==========================================
# 1. Global Definitions (Single Source of Truth)
# ==========================================
normalization_dict = {
    "Canadian Civil Liberties Association": "CCLA",
    "Association canadienne des libertés civiles": "CCLA",
    "British Columbia Civil Liberties Association": "BCCLA",
    "Women's Legal Education and Action Fund": "LEAF",
    "Women's Legal Education and Action Fund Inc.": "LEAF",
    "Women’s Legal Education and Action Fund (LEAF)": "LEAF",
    "Fonds d'action et d'éducation juridiques pour les femmes (FAEJ)": "LEAF",
    "Amnesty International Canadian Section (English Speaking)": "Amnesty International",
    "Amnesty International (Canadian Section, English Branch)": "Amnesty International",
    "Egale Canada Human Rights Trust": "Egale Canada", 
    "Criminal Lawyers' Association (Ontario)": "CLA",
    "Criminal Lawyers' Association of Ontario": "CLA",
    "Criminal Lawyers' Association": "CLA",
    "Association québécoise des avocats et avocates de la défense": "AQAAD",
}

state_keywords = [
    'attorney general', 'procureur général', 'director of public prosecutions',
    'directeur des poursuites', 'her majesty', 'his majesty', 'ministre', 
    'minister', 'information and privacy commissioner'
]

def clean_counsel_string(counsel_str):
    s = re.sub(r',\s*Q\.?C\.?', ' (QC)', counsel_str, flags=re.IGNORECASE)
    s = re.sub(r',\s*K\.?C\.?', ' (KC)', s, flags=re.IGNORECASE)
    s = re.sub(r',\s*Ad\.?\s*E\.?', ' (AdE)', s, flags=re.IGNORECASE)
    s = re.sub(r',\s*c\.?r\.?', ' (cr)', s, flags=re.IGNORECASE)
    s = re.sub(r',\s*LSM', ' (LSM)', s, flags=re.IGNORECASE)
    return s

def clean_name(name):
    return name.replace("'", "’").replace('"', '”').strip()

def get_docket(text):
    match = re.search(r'(?:File No\.|No du greffe|dossier)\s*[:\.]?\s*(\d{5})', str(text), re.IGNORECASE)
    if match: return match.group(1)
    fallback = re.search(r'\b(\d{5})\b', str(text)[:1000])
    return fallback.group(1) if fallback else None

def extract_all_charter_sections(text):
    matches = re.findall(r'\b(?:s\.|ss\.|sec\.|section|sections)\s*(\d{1,2}(?:\([a-z1-9]\))?)', str(text).lower())
    valid_sections = []
    for m in matches:
        base_num_match = re.match(r'(\d+)', m)
        if base_num_match and 1 <= int(base_num_match.group(1)) <= 35:
            valid_sections.append(f"s.{m.strip().replace(' ', '')}")
    return list(set(valid_sections)) if valid_sections else ['Unspecified']

# ==========================================
# 2. Extract Charter Sections & Merge with Fixed Dataset
# ==========================================
print("📥 Fetching SCC texts from HuggingFace to extract Charter Sections...")
dataset = load_dataset("a2aj/canadian-case-law", data_dir="SCC", split="train")
df_raw = dataset.to_pandas()
df_raw['Docket_Number'] = df_raw['unofficial_text_en'].apply(get_docket).astype(str)
df_raw['All_Sections'] = df_raw['unofficial_text_en'].apply(extract_all_charter_sections)

# ⚠️ Note: we are using the FIXED Full Dataset after accuracy check and error correction
print("🔗 Merging with pristine Intervenor data...")
df_intervenors = pd.read_csv("A2J_Intervenors_Full_Dataset_FIXED.csv")
df_intervenors['Docket_Number'] = df_intervenors.get('Docket', df_intervenors.get('Docket_Number', pd.Series())).astype(str) 
df_merged = pd.merge(df_intervenors, df_raw[['Docket_Number', 'All_Sections']], on='Docket_Number', how='left')

# Output 1
df_merged.to_csv("A2J_Cases_with_Charter_Sections.csv", index=False)

# ==========================================
# 3. The Master Loop: Parsing One Time for Three Outputs
# ==========================================
print("⚙️ Processing raw text and building matrices (One-Pass Routing)...")

edge_records = []
ngo_section_records = []
counsel_section_records = []

for index, row in df_merged.iterrows():
    text = row['Intervenors_and_Counsel']
    sections = row['All_Sections']
    
    # Section Lists
    if isinstance(sections, str):
        try: sections = ast.literal_eval(sections)
        except: sections = []
    elif not isinstance(sections, list):
        sections = []

    if pd.isna(text) or "No Intervenors" in text:
        continue
        
    lines = str(text).split('\n')
    for line in lines:
        if '->' in line:
            parts = line.split('->')
            raw_org = parts[0].replace('[Intervener]', '').strip()
            raw_counsel = parts[1].strip()
            
            # Clean State Actors
            if any(kw in raw_org.lower() for kw in state_keywords):
                continue
                
            org = normalization_dict.get(raw_org, raw_org)
            
            # For NGO vs Sections
            for sec in sections:
                ngo_section_records.append({'NGO': org, 'Charter_Section': sec})
            
            # Clean Counsel's Name
            if raw_counsel and raw_counsel != "None listed":
                cleaned_c_str = clean_counsel_string(raw_counsel)
                lawyers = [clean_name(l) for l in cleaned_c_str.split(',')]
                
                for lawyer in lawyers:
                    if not lawyer: continue
                    
                    # For Resource Map
                    edge_records.append({'NGO': org, 'Counsel': lawyer})
                    
                    # For Counsel vs Sections
                    for sec in sections:
                        if sec not in ['s.Unspecified', 'Unspecified']:
                            counsel_section_records.append({'Counsel': lawyer, 'Charter_Section': sec})

# ==========================================
# 4. Generate & Save the Final Matrices
# ==========================================
print("💾 Generating final DataFrames and saving CSVs...")

# Output 2: Network Edges 
df_edges = pd.DataFrame(edge_records)
df_edges_grouped = df_edges.groupby(['NGO', 'Counsel']).size().reset_index(name='Weight')
df_edges_grouped.to_csv("A2J_Network_Edges.csv", index=False)

# Output 3: NGO vs Charter Sections Matrix 
df_ngo_sec = pd.DataFrame(ngo_section_records)
pivot_ngo = pd.crosstab(df_ngo_sec['NGO'], df_ngo_sec['Charter_Section'])
pivot_ngo['Total_Appearances'] = pivot_ngo.sum(axis=1)
pivot_ngo = pivot_ngo.sort_values(by='Total_Appearances', ascending=False)
pivot_ngo.to_csv("A2J_NGO_vs_All_Sections_Matrix.csv")

# Output 4: Counsel vs Charter Sections Matrix 
df_counsel_sec = pd.DataFrame(counsel_section_records)
pivot_counsel = pd.crosstab(df_counsel_sec['Counsel'], df_counsel_sec['Charter_Section'])
pivot_counsel.to_csv("A2J_Counsel_vs_All_Sections_Matrix.csv")

print("\n🎉 Unified Pipeline Complete! All 4 CSVs have been successfully generated.")

# Appendix C: Accuracy Check and Error Correction

##### C.1: Sample of 30 cases selection and output

In [ ]:
import pandas as pd

# 1. Load the Full Dataset
df = pd.read_csv("A2J_Intervenors_Full_Dataset.csv")

# 2. Randomly select 30 cases
validation_sample = df.sample(n=30, random_state=42).copy()

# 3. Specify the column names to export
columns_to_export = ['Case_Name', 'Intervenors_and_Counsel', 'SCC_Registry_URL']

# 4. Safe export mechanism (automatically skips non-existent columns to prevent errors)
actual_columns = []
for col in columns_to_export:
    if col in validation_sample.columns:
        actual_columns.append(col)
    else:
        print(f"⚠️ Warning: couldn't find a column named ‘{col}’ in the table - will skip it")

validation_sample[actual_columns].to_csv("Accuracy_Check_Sample_30.csv", index=False)

print("\n✅ Validation sample generated successfully!")
print("👉 Accuracy_Check_Sample_30.csv created")

#### C.2: Error Correction for the Full Dataset

In [ ]:
import pandas as pd

print("🛠️ Initiating Targeted Patching for missing counsel...")

# 1. Load the Full Dataset
df = pd.read_csv("A2J_Intervenors_Full_Dataset.csv")

# 2. Locate the cases with "None listed"
mask = df['Intervenors_and_Counsel'].str.contains("None listed", na=False)
missing_cases = df[mask]

print(f"🚨 Found {len(missing_cases)} mega-cases with 'None listed' error.")
print("The cases are:")
for idx, row in missing_cases.iterrows():
    docket = row.get('Docket_Number', 'Unknown Docket')
    case_name = row.get('Case_Name', 'Unknown Case')
    print(f" - [{docket}] {case_name}")

# ==========================================
# 3. Ground Truth Patch

patch_dictionary = {
    
    35226: """[Intervener] Couche-Tard Inc. -> Louis-Martin O'Neill, Jean-Philippe Groleau
[Intervener] Alimentation Couche-Tard Inc., Dépan-Escompte Couche-Tard Inc. -> Louis-Martin O'Neill, Jean-Philippe Groleau
[Intervener] Bonin, Céline -> Louis Belleau, Ad. E., André Mignault, Luc Jobin
[Intervener] Ultramar Ltd -> Louis P. Bélanger, Q.C., Caroline Plante, Julie Girard
[Intervener] Pétroles Therrien Inc., Distributions Pétrolières Therrien Inc. -> Pascale Cloutier, Fadi Amine
[Intervener] Irving Oil Inc. / Irving Oil Operations Ltd. -> Sylvain Lussier, Elizabeth Meloche
[Intervener] Olco Petroleum Group Inc. -> Éric Vallières, Sidney Elbaz, Rachel April-Giguère
[Intervener] Philippe Gosselin & Associés Ltd -> Michel Chabot, Hugo Poirier
[Intervener] André Bilodeau, Carol Lehoux, Claude Bédard, Stéphane Grant -> Michel Chabot, Hugo Poirier
[Intervener] Coop fédérée, Robert Murphy, Gary Neiderer -> Julie Chenette, Sébastien Pierre-Roy
[Intervener] 9142-0935 Québec Inc., 9131-4716 Québec Inc., Groupe Denis Mongeau Inc. -> Michel Jolin, Marie-Geneviève Masson, Favrice Anglade Vil
[Intervener] Couturier, Luc -> Roxanne Hardy
[Intervener] Forget, Luc -> Roxanne Hardy
[Intervener] Angers, Guy -> Jean-Rémi Thibault
[Intervener] Ouellet, Jacques -> Jean-Rémi Thibault
[Intervener] Pétroles Cadrin Inc., Daniel Drouin -> Daniel O'Brien, Pierre Grégoire
[Intervener] Pétroles Global inc. / Global Fuels Inc. -> David Quesnel
[Intervener] Pétroles Global (Québec) inc. / Global Fuels (Quebec) Inc. -> David Quesnel
[Intervener] Provigo Distribution Inc. -> Robert E. Charbonneau, Tommy Tremblay, Anne Merminod
[Intervener] Aubut, Carole -> Richard Morin
[Intervener] Bédard, Richard -> Mark J. Paci
[Intervener] Payette, Christian -> Dominic Desjarlais, Gérald Soulière, Julie Philippe
[Intervener] Bourassa, Pierre -> Jean Berthiaume, Richard Mallette
[Intervener] Leblond, Daniel -> Jo-Anne Demers, Jean-Olivier Lessard
[Intervener] Dépanneur Magog-Orford Inc -> Geneviève Cotnam, Geneviève Allen, Émilie Bilodeau
[Intervener] 2944-4841 Québec Inc. -> Charles Gosselin
[Intervener] Société coopérative agricole des Bois-Francs -> Claude Brulotte
[Intervener] Gestion Astral Inc., Lise Delisle -> Maryse Carrier, Jean-François Côté
[Intervener] 134553 Canada Inc. -> Benoît Lapointe, Maxime Nasr
[Intervener] Garage Luc Fecteau et fils Inc., Station-Service Jacques Blais Inc., 9029-6815 Québec Inc., Garage Jacques Robert Inc. -> Jean-Claude Chabot, Claudia Marie Chabot
[Intervener] Gérald Groulx Station Service Inc., Services Autogarde D.D. Inc., 9010-1460 Québec Inc. -> Stéphane Reynolds
[Intervener] Armand Pouliot, Julie Roberge, Station-Service Pouliot et Roberge s.e.n.c. -> Pierre Paradis, Anne-Marie Lessard
[Intervener] 9038-6095 Québec Inc. -> Marcel Després
[Intervener] 9083-0670 Québec Inc., Gestion Ghislain Lallier Inc. -> Sylvain Beauregard, Claude A. Roy
[Intervener] 2429-7822 Québec Inc. -> Simon Letendre
[Intervener] 2627-3458 Québec Inc. -> Sylvain Beauregard, Claude A. Roy
[Intervener] 9098-0111 Québec Inc. -> Guy Plourde
[Intervener] 2311-5959 Québec Inc., Gaz-O-Pneus Inc. -> Pierre Lessard
[Intervener] C. Lagrandeur et fils Inc. -> Jean Beaudry
[Intervener] Universy Galt Service Inc. -> Simon Letendre
[Intervener] Valérie Houde, Sylvie Fréchette, Robert Beaurivage -> Maxime Bernatchez
[Intervener] 9011-4653 Québec Inc., Pétroles Remay Inc., Variétés Jean Yves Plourde Inc. and 9016-8360 Québec Inc. -> None listed
[Intervener] Benoît, France -> Pascale Cloutier, Fadi Amine
[Intervener] Michaud, Richard -> Pascale Cloutier, Fadi Amine
[Intervener] Attorney General of Ontario -> Megan Stephens, Deborah Calderwood""",

    37208: """[Intervener] Zulkoskey, Tania -> Stephen J. Moreau, Nadia Lambek
[Intervener] Income Security Advocacy Centre, Sudbury Community Legal Clinic, Chinese and Southeast Asian Legal Clinic, Community Legal Assistance Society and HIV & AIDS Legal Clinic Ontario -> Marie Chen, Jackie Esmonde
[Intervener] Canadian Muslim Lawyers Association -> Kumail Karimjee, Nabila F. Qureshi
[Intervener] Council of Canadians with Disabilities -> Kerri Joffe, Dianne Wintermute
[Intervener] Women's Legal Education and Action Fund, Native Women's Association of Canada -> Mary Eberts, Kim Stanton, K.R. Virginia Lomax
[Intervener] Amnesty International -> Justin Safayeni, Stephen Alyward
[Intervener] First Nations Child and Family Caring Society of Canada -> David P. Taylor
[Intervener] Matson, Jeremy E. -> None listed
[Intervener] African Canadian Legal Clinic -> Tamara Thomas, Faisal Mirza
[Intervener] Aboriginal Legal Services -> Emily Hill, Emilie Lahaie
[Intervener] Attorney General of Quebec -> Amélie Pelletier Desrosiers
[Intervener] Public Service Alliance of Canada -> Andrew Raven, Andrew Astritis, Morgan Rowe""",

    37748: """[Intervener] Attorney General of Ontario -> Sara Blake, Judie Im
[Intervener] Canadian Council for Refugees -> Jamie Liew, Gerald Heckman, Jean Lash
[Intervener] Advocacy Centre for Tenants Ontario -> Karen Andrews
[Intervener] Ontario Securities Commission, British Columbia Securities Commission and Alberta Securities Commission -> Matthew H. Britton, Jennifer M. Lynch, Paloma Ellard, David Hainey, Don Young
[Intervener] Ecojustice Canada Society -> Laura Bowman, Bronwyn Roe
[Intervener] Workplace safety and Insurance Appeals Tribunal (Ontario) -> Michelle Alton, David Corbett, Kayla Seyler, Ana Rodriguez
[Intervener] Workers' Compensation Appeals Tribunal (Northwest Territories and Nunavut) and Workers' Compensation Appeals Tribunal (Nova Scotia) -> Michelle Alton, David Corbett, Kayla Seyler, Ana Rodriguez
[Intervener] Attorney General for Saskatchewan -> Laura Mazenc, Kyle McCreary, Johnna Van Parys
[Intervener] British Columbia International Commercial Arbitration Centre Foundation -> Gavin R. Cameron, Tom Posyniak
[Intervener] Council of Canadian Administrative Tribunals -> Terrence J. O'Sullivan, Paul Michell, James Renihan
[Intervener] Ontario Labour-Management Arbitrators' Association and Conférence des arbitres du Québec -> Linda R. Rothstein, Michael Fenrick, Angela E. Rae, Anne Marie Heenan
[Intervener] National Academy of Arbitrators -> Susan L. Stewart
[Intervener] Canadian Labour Congress -> Steven M. Barrett, Ethan Poskander, Daniel Sheppard
[Intervener] Attorney General of Quebec -> Stéphane Rochette
[Intervener] National Association of Pharmacy Regulatory Authorities -> William W. Shores, Q.C., Kirk N. Lambrecht, Q.C.
[Intervener] Queen's Prison Law Clinic -> Brendan Van Niejenhuis, Andrea Gonsalves
[Intervener] Attorney General of British Columbia -> Micah Rankin, Leah Greathead, Katie Hamilton, J. Gareth Morley
[Intervener] Advocates for the Rule of Law -> Adam Goldenberg, Robyn Gifford, Asher Honickman
[Intervener] Parkdale Community Legal Services -> Toni Schweitzer, Ronald Poulton
[Intervener] Cambridge Comparative Administrative Law Forum -> Bruno Gélinas-Faucher, Paul Warchuk, Francis Lévesque
[Intervener] Appeals Commission for Alberta Workers' Compensation and Workers' Compensation Appeals Tribunal (New Brunswick) -> Michelle Alton, David Corbett, Kayla Seyler, Ana Rodriguez
[Intervener] Samuelson-Glushko Canadian Internet Policy and Public Interest Clinic -> Alyssa Tomkins, James Plotkin, Michel Bastarache
[Intervener] Canadian Bar Association -> Jonathan M. Coady, Justin L. Milne
[Intervener] Canadian Association of Refugee Lawyers -> Anthony Navaneelan, Audrey Macklin
[Intervener] Community & Legal Aid Services Programme -> Subodh Bharati, David Cote
[Intervener] Association québécoise des avocats et avocates en droit de l'immigration -> Peter Shams, Claudia Andrea Molina, Guillaume Cliche-Rivard, David Berger
[Intervener] First Nations Child and Family Caring Society of Canada -> David P. Taylor, Sarah Clarke""",

    38663: """[Intervener] Attorney General of British Columbia -> J. Gareth Morley, Jacqueline D. Hughes
[Intervener] Attorney General of New Brunswick -> Rachelle Standing, Isabel Lavoie Daigle
[Intervener] Attorney General of Ontario -> Joshua Hunter, Padraic Ryan, Otto Ranalli
[Intervener] Saskatchewan Power Corporation and SaskEnergy Incorporated -> David M. A. Stack, Q.C.
[Intervener] Canadian Taxpayers Federation -> R. Bruce E. Hallsor, Hana Felix
[Intervener] International Emissions Trading Association -> Elisabeth DeMarco, Jonathan McGillivray
[Intervener] Canadian Public Health Association -> Jennifer L. King, Michael Finley, Liane Langstaff
[Intervener] Athabasca Chipewyan First Nation -> Amir Attaran, Matt Hulse
[Intervener] Canadian Environmental Law Association, Environmental Defence Canada Inc. and Sisters of Providence of St. Vincent de Paul -> Joseph F. Castrilli, Theresa McClenaghan, Richard D. Lindgren
[Intervener] Assembly of First Nations -> Stuart Wuttke, Julie McGregor, Adam Williamson, Victor Carter
[Intervener] David Suzuki Foundation -> Joshua Ginsberg, Randy Christensen
[Intervener] Canada's Ecofiscal Commission -> Stewart Elgie, LSM
[Intervener] Climate Justice Saskatoon, National Farmers Union, Saskatchewan Coalition for Sustainable Development, Saskatchewan Council for International Cooperation, Saskatchewan Environmental Society, SaskEV -> Larry W. Kowalchuk, Taylor-Anne Yee, Jonathan Stockdale
[Intervener] Council of Canadians: Prairie and Northwest Territories Region, Council of Canadians: Regina Chapter, Council of Canadians: Saskatoon Chapter, New-Brunswick Anti-Shale Gas Alliance and Youth of the Earth -> Larry W. Kowalchuk
[Intervener] Attorney General of Alberta -> Peter A. Gall, K.C., Benjamin J. Oliphant, Steven A. A. Dollansky, L. Christine Enns, Q.C.
[Intervener] Attorney General of Manitoba -> Michael Conner, Allison Kindle Pejovic
[Intervener] Attorney General of Quebec -> Jean-Vincent Lacroix, Laurie Anctil
[Intervener] Progress Alberta Communications Limited -> Avnish Nanda, Martin Olszynski
[Intervener] Canadian Labour Congress -> Simon Archer, Mariam Moktar, Daniel Sheppard
[Intervener] Oceans North Conservation Society -> David W.L. Wu
[Intervener] Amnesty International Canada -> Justin Safayeni, Zachary Al-Khatib
[Intervener] National Association of Women and the Law and Friends of the Earth -> Nathalie Chalifour, Anne Levesque
[Intervener] Smart Prosperity Institute -> Jeremy de Beer, Guy Régimbald
[Intervener] Centre québécois du droit de l'environnement et Équiterre -> David Robitaille, Marc Bishai
[Intervener] Generation Squeeze, Public Health Association of British Columbia, Saskatchewan Public Health Association, Canadian Association of Physicians for the Environment, Canadian Coalition for the Rights of the Child and Youth Climate Lab -> Nathan Hume, Emma Hume, Cam Brewer
[Intervener] Assembly of Manitoba Chiefs -> Joëlle Pastora Sala, Byron Williams, Katrine Dilay
[Intervener] City of Richmond, City of Victoria, City of Nelson, District of Squamish, City of Rossland and City of Vancouver -> Paul A. Hildebrand, Andrew Carricato
[Intervener] Thunderchild First Nation -> Dusty T. Ernewein, Lorne R. Fagnan""",

    40061: """[Intervener] Attorney General of Manitoba -> Deborah L. Carlson, Kathryn Hart, Heather Leonoff, K.C.
[Intervener] Attorney General of British Columbia -> Leah Greathead
[Intervener] Grand Council of Treaty #3 -> Robert Janes, K.C., Naomi Moses
[Intervener] Innu Takuaikan Uashat Mak Mani-Utenam (ITUM), agissant comme bande traditionnelle et au nom des Innus de Uashat Mak Mani-Utenam -> James A. O'Reilly, Ad.E., Marie-Claude André-Grégoire, Michelle Corbu, Vincent Carney
[Intervener] Federation of Sovereign Indigenous Nations -> Michael Seed, Nicholas Dodd, Rosa Victoria Adams
[Intervener] Peguis Child and Family Services -> Hafeez Khan, Earl C. Stevenson
[Intervener] Native Women's Association of Canada -> Sarah Niman, Kira Poirier
[Intervener] Council of Yukon First Nations -> Tammy Shoranick, Daryn Leas, James M. Coady
[Intervener] Indigenous Bar Association in Canada -> Paul Seaman, Keith Brown
[Intervener] Chiefs of Ontario -> Maggie Wente, Krista Nerland
[Intervener] Inuvialuit Regional Corporation -> Katherine Hensel, Kristie Tsang, Todd Orvitz
[Intervener] Inuit Tapiriit Kanatami, Nunatsiavut Government and Nunavut Tunngavik Incorporated -> Brian A. Crane, K.C., Graham Ragan, Alyssa Flaherty-Spence, Kate Darling
[Intervener] NunatuKavut Community Council -> Jason Cooke, Ashley Hamp-Gonsalves
[Intervener] Lands Advisory Board -> William B. Henderson
[Intervener] Métis National Council, Métis Nation-Saskatchewan, Métis Nation of Alberta, Métis Nation British Columbia, Métis Nation of Ontario and Les femmes Michif Otipemisiwak -> Jason T. Madden, Alexander DeParde, Emilie N. Lahaie
[Intervener] Listuguj Mi'Gmaq Government -> Zachary Davis, Riley Weyman
[Intervener] Congress of Aboriginal Peoples -> Andrew K. Lokan, Glynnis Hawe
[Intervener] First Nations Family Advocate Office -> Joëlle Pastora Sala, Allison Fenske, Maximilian Griffin-Rill, Adrienne Cooper
[Intervener] Assembly of Manitoba Chiefs -> David Outerbridge, Craig Gilchrist, Rebecca Amoah
[Intervener] First Nations of the Maa-Nulth Treaty Society -> Maegen M. Giltrow, K.C., Natalia Sudeyko, Lisa Glowacki
[Intervener] Tribal Chiefs Ventures Inc. -> Aaron Christoff, Brent Murphy
[Intervener] Union of British Columbia Indian Chiefs, First Nations Summit of British Columbia and British Columbia Assembly of First Nations -> Gib van Ert, Fraser Harland, Mary Ellen Turpel-Lafond
[Intervener] David Asper Centre for Constitutional Rights -> Jessica Orkin, Natai Shelsen
[Intervener] Regroupement Petapan -> François G. Tremblay, Benoît Amyot
[Intervener] Canadian Constitution Foundation -> Jesse Hartery, Simon Bouthillier, Allison Spiegel
[Intervener] Carrier Sekani Family Services Society, Cheslatta Carrier Nation, Nadleh Whuten, Saik'uz First Nation and Stellat'en First Nation -> Scott A. Smith
[Intervener] Conseil des Atikamekw d'Opitciwan -> Kevin Ajmo, Frédéric Boily, Stéphanie Ajmo, Jean-François Delisle
[Intervener] Vancouver Aboriginal Child and Family Services Society -> Maxime Faille
[Intervener] Nishnawbe Aski Nation -> Julian N. Falconer, Christopher Rapson, Mitchell Goldenberg
[Intervener] Attorney General of the Northwest Territories -> Trisha Paradis, Sandra Jungles"""
}

# ==========================================
# 4. The Bulletproof Injector
# ==========================================
patches_applied = 0

for docket_num, correct_text in patch_dictionary.items():
    # Insert complete docket
    clean_docket_col = df['Docket_Number'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
    clean_target_docket = str(docket_num).strip()
    target_row = clean_docket_col == clean_target_docket
    
    if target_row.any():
        df.loc[target_row, 'Intervenors_and_Counsel'] = correct_text.strip()
        patches_applied += 1
        print(f"✅ Patched Docket: {clean_target_docket}")
    else:
        # If cannot find
        print(f"⚠️ Warining: cannot find {clean_target_docket}！")

print(f"\n🎉 Successfully applied {patches_applied} patches!")

# 5. Save fixed Full Dataset
fixed_filename = "A2J_Intervenors_Full_Dataset_FIXED.csv"
df.to_csv(fixed_filename, index=False)
print(f"📁 The pristine dataset is now saved as: {fixed_filename}")